# SolarSDE Notebook 5 — Multi-Seed Runs, CTI Analysis, Economic Value, Figures

**Runtime:** ~2-4 hours on Colab T4 / Kaggle P100

**Prerequisite:** Notebooks 1-4 completed.

**This notebook:**
1. Re-trains SolarSDE (SDE + Score Decoder) with seeds 123 and 456 for variance estimates
2. Computes 3-seed mean ± std for the main results table
3. CTI analysis: Spearman correlation vs GHI variability, CRPS by CTI quartile, K-means regime clustering
4. Economic value simulation (annual reserve cost savings)
5. Reliability diagram data + PIT histogram
6. Generates all paper figures
7. Saves final formatted tables


In [ ]:
# ==== Install dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("pvlib", "properscoring", "pyarrow", "tqdm")
print("Dependencies installed.")


In [ ]:
# ==== Environment Detection & Setup ====
import os, sys, time, json, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

PERSIST_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PERSIST_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LATENT_DIR = PERSIST_DIR / "latents"
LATENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")
print(f"Working directory:  {WORK_DIR}")


In [ ]:
# ==== GPU Check ====
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:          {torch.cuda.get_device_name(0)}")
    print(f"Memory:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: Running on CPU — will be slow. Change Runtime > Change runtime type > GPU")
print(f"Using device: {DEVICE}")


## Fast-start — pull Notebook 1 outputs from GitHub (skips VAE retraining)

In [ ]:
# ==== Fast-start: fetch Notebook 1 outputs from GitHub if persistent storage is empty ====
# This lets Notebooks 2-5 run standalone without having re-executed Notebook 1.
GITHUB_REPO = "https://github.com/keshavkrishnan08/SDE"
GITHUB_RAW = "https://raw.githubusercontent.com/keshavkrishnan08/SDE/main"

need_nb1_outputs = not (CHECKPOINT_DIR / "vae_best.pt").exists() or not any(LATENT_DIR.glob("train_*.npy"))
if need_nb1_outputs:
    print("Notebook 1 outputs not found in persistent storage — pulling from GitHub ...")
    import requests
    files_to_pull = [
        ("checkpoints/vae_best.pt",          CHECKPOINT_DIR / "vae_best.pt"),
        ("checkpoints/vae_final.pt",         CHECKPOINT_DIR / "vae_final.pt"),
        ("results/vae_training_history.csv", RESULTS_DIR / "vae_training_history.csv"),
        ("splits/train.parquet",             PERSIST_DIR / "splits" / "train.parquet"),
        ("splits/val.parquet",               PERSIST_DIR / "splits" / "val.parquet"),
        ("splits/test.parquet",              PERSIST_DIR / "splits" / "test.parquet"),
    ]
    for split in ["train", "val", "test"]:
        for k in ["latents", "cti", "ghi", "covariates", "is_ramp"]:
            files_to_pull.append((f"latents/{split}_{k}.npy", LATENT_DIR / f"{split}_{k}.npy"))

    for rel, dest in files_to_pull:
        url = f"{GITHUB_RAW}/colab_outputs/{rel}"
        if dest.exists() and dest.stat().st_size > 0:
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(url, timeout=120)
            if r.status_code == 200 and len(r.content) > 100:
                dest.write_bytes(r.content)
                print(f"  OK  {rel}  ({len(r.content)/1e6:.2f} MB)")
            else:
                print(f"  SKIP {rel}  (status {r.status_code})")
        except Exception as e:
            print(f"  FAIL {rel}: {e}")
    print("Fast-start pull complete.")
else:
    print("Notebook 1 outputs already present in persistent storage.")


In [ ]:
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy import stats
from sklearn.cluster import KMeans
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time, json

def load_split(s):
    return {
        "Z": np.load(LATENT_DIR / f"{s}_latents.npy"),
        "cti": np.load(LATENT_DIR / f"{s}_cti.npy"),
        "ghi": np.load(LATENT_DIR / f"{s}_ghi.npy"),
        "cov": np.load(LATENT_DIR / f"{s}_covariates.npy"),
        "ramp": np.load(LATENT_DIR / f"{s}_is_ramp.npy"),
    }
data = {s: load_split(s) for s in ["train", "val", "test"]}
Z_DIM = data["train"]["Z"].shape[1]; C_DIM = max(1, data["train"]["cov"].shape[1])
HORIZONS = [6, 30, 60, 120, 180]; HORIZON_MIN = {6:1, 30:5, 60:10, 120:20, 180:30}
N_SAMPLES = 50; N_EVAL = min(1000, len(data["test"]["Z"]) - max(HORIZONS) - 1)

FIGURES_DIR = PERSIST_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==== Neural SDE model ====
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class DriftNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim + 1 + c_dim, h), nn.SiLU(inplace=True),
            nn.Linear(h, h), nn.SiLU(inplace=True),
            ResBlock(h), ResBlock(h),
            nn.Linear(h, z_dim),
        )
    def forward(self, z, t, c):
        return self.net(torch.cat([z, t, c], dim=-1))

class CTIDiffNet(nn.Module):
    """CTI-gated diffusion: σ = Softplus(MLP(z) * Softplus(MLP(CTI)))"""
    def __init__(self, z_dim=64, h=64):
        super().__init__()
        self.cti_gate = nn.Sequential(nn.Linear(1, h), nn.Softplus())
        self.state = nn.Sequential(nn.Linear(z_dim, h), nn.SiLU(inplace=True))
        self.out = nn.Sequential(nn.Linear(h, z_dim), nn.Softplus())
    def forward(self, z, cti):
        return self.out(self.state(z) * self.cti_gate(cti))

class LatentNeuralSDE(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, drift_h=256, diff_h=64, lambda_sigma=1.0):
        super().__init__()
        self.z_dim = z_dim
        self.lambda_sigma = lambda_sigma
        self.drift = DriftNet(z_dim, c_dim, drift_h)
        self.diffusion = CTIDiffNet(z_dim, diff_h)
    def forward(self, z, t, c, cti):
        return self.drift(z, t, c), self.diffusion(z, cti)
    def sde_matching_loss(self, z, zn, t, c, cti, dt=1.0):
        mu = self.drift(z, t, c)
        sigma = self.diffusion(z, cti)
        dz = (zn - z) / dt
        drift_l = F.mse_loss(mu, dz)
        resid = (zn - z - mu * dt).pow(2) / dt
        diff_l = F.mse_loss(sigma.pow(2), resid)
        return {"loss": drift_l + self.lambda_sigma * diff_l,
                "drift": drift_l, "diffusion": diff_l}


In [ ]:
# ==== Conditional Score-Matching Irradiance Decoder (CSMID) ====
class ScoreRes(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d), nn.SiLU(inplace=True), nn.Linear(d, d))
    def forward(self, x): return x + self.net(x)

class ScoreNet(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2):
        super().__init__()
        d_in = 1 + 1 + z_dim + 1 + c_dim
        layers = [nn.Linear(d_in, h), nn.SiLU(inplace=True)]
        for _ in range(blocks): layers.append(ScoreRes(h))
        layers.append(nn.Linear(h, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, g, s, z, cti, c):
        return self.net(torch.cat([g, s, z, cti, c], dim=-1))

class CondScoreDecoder(nn.Module):
    def __init__(self, z_dim=64, c_dim=5, h=256, blocks=2, steps=100, b0=1e-4, b1=0.02):
        super().__init__()
        self.steps = steps
        self.score = ScoreNet(z_dim, c_dim, h, blocks)
        betas = torch.linspace(b0, b1, steps)
        alphas = 1 - betas
        ac = torch.cumprod(alphas, dim=0)
        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alphas_cum", ac)
        self.register_buffer("sac", torch.sqrt(ac))
        self.register_buffer("s1mac", torch.sqrt(1 - ac))
    def q(self, g0, si, eps):
        return self.sac[si].unsqueeze(-1) * g0 + self.s1mac[si].unsqueeze(-1) * eps
    def training_loss(self, g0, z, cti, c):
        B = g0.shape[0]
        si = torch.randint(0, self.steps, (B,), device=g0.device)
        sn = (si.float() / self.steps).unsqueeze(-1)
        eps = torch.randn_like(g0)
        gs = self.q(g0, si, eps)
        pred = self.score(gs, sn, z, cti, c)
        target = -eps / self.s1mac[si].unsqueeze(-1)
        return {"loss": F.mse_loss(pred, target)}
    @torch.no_grad()
    def sample(self, z, cti, c, n=1):
        B = z.shape[0]
        z_e = z.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        cti_e = cti.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        c_e = c.unsqueeze(1).expand(B, n, -1).reshape(B * n, -1)
        x = torch.randn(B * n, 1, device=z.device)
        for i in reversed(range(self.steps)):
            sn = torch.full((B * n, 1), i / self.steps, device=z.device)
            score = self.score(x, sn, z_e, cti_e, c_e)
            b, a, ac = self.betas[i], self.alphas[i], self.alphas_cum[i]
            mean = (1 / a.sqrt()) * (x + b * score * (1 - ac).sqrt())
            if i > 0:
                x = mean + b.sqrt() * torch.randn_like(x)
            else:
                x = mean
        return x.view(B, n)


In [ ]:
# ==== Probabilistic forecasting metrics ====
def crps_empirical(y_true, y_samples):
    """CRPS from empirical samples. y_true: (N,), y_samples: (N, M)."""
    N, M = y_samples.shape
    t1 = np.mean(np.abs(y_samples - y_true[:, None]), axis=1)
    ys = np.sort(y_samples, axis=1)
    w = 2 * np.arange(1, M + 1) - M - 1
    t2 = np.sum(w[None, :] * ys, axis=1) / (M * M)
    return t1 - t2

def picp(y_true, y_samples, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float(((y_true >= lo) & (y_true <= hi)).mean())

def pinaw(y_samples, y_range, alpha=0.9):
    lo = np.quantile(y_samples, (1 - alpha) / 2, axis=1)
    hi = np.quantile(y_samples, 1 - (1 - alpha) / 2, axis=1)
    return float((hi - lo).mean() / max(y_range, 1e-9))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def all_metrics(y_true, y_samples, is_ramp=None, alpha=0.9):
    y_med = np.median(y_samples, axis=1)
    y_range = y_true.max() - y_true.min()
    crps = crps_empirical(y_true, y_samples)
    out = {
        "crps": float(crps.mean()),
        "picp": picp(y_true, y_samples, alpha),
        "pinaw": pinaw(y_samples, y_range, alpha),
        "rmse": rmse(y_true, y_med),
        "mae":  mae(y_true, y_med),
    }
    if is_ramp is not None and is_ramp.sum() > 0:
        out["ramp_crps"] = float(crps[is_ramp].mean())
    return out


In [ ]:
# ==== Euler-Maruyama SDE solver with stability clamping ====
# The Neural SDE is trained on 1-step dynamics (SDE Matching); without clamping,
# long-horizon rollouts can drift out of the VAE latent distribution and the drift
# MLP will extrapolate catastrophically. We clamp per-step drift, diffusion, and
# the state vector to stay within ~4σ of training latent statistics.

# Compute training-latent stats once at module load.
_train_Z_np = np.load(LATENT_DIR / "train_latents.npy")
Z_MEAN = torch.from_numpy(_train_Z_np.mean(0)).float().to(DEVICE)
Z_STD  = torch.from_numpy(_train_Z_np.std(0)).float().to(DEVICE) + 1e-6
Z_CLAMP_STDS = 4.0
MU_CAP = 5.0
SIGMA_CAP = 2.0

def em_step(drift_fn, diff_fn, z, t, c, cti, dt):
    mu = drift_fn(z, t, c).clamp(-MU_CAP, MU_CAP)
    sigma = diff_fn(z, cti).clamp(0.0, SIGMA_CAP)
    z_new = z + mu * dt + sigma * (dt ** 0.5) * torch.randn_like(z)
    return torch.clamp(z_new, Z_MEAN - Z_CLAMP_STDS * Z_STD, Z_MEAN + Z_CLAMP_STDS * Z_STD)

def solve_sde_horizons(sde, z0, horizons, c, cti, N=50, dt=1.0):
    B, d = z0.shape
    mx = max(horizons)
    hset = set(horizons)
    z = z0.unsqueeze(1).expand(B, N, d).reshape(B * N, d)
    c_e = c.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    cti_e = cti.unsqueeze(1).expand(B, N, -1).reshape(B * N, -1)
    out = {}
    for step in range(mx):
        t = torch.full((B * N, 1), float(step), device=z0.device)
        z = em_step(sde.drift, sde.diffusion, z, t, c_e, cti_e, dt)
        if (step + 1) in hset:
            out[step + 1] = z.view(B, N, d).clone()
    return out


In [ ]:
class LatentSeqDataset(Dataset):
    def __init__(self, d):
        self.Z=d["Z"]; self.cti=d["cti"]; self.ghi=d["ghi"]; self.cov=d["cov"]
    def __len__(self): return max(0, len(self.Z) - 1)
    def __getitem__(self, i):
        return {"z_t": torch.from_numpy(self.Z[i]).float(),
                "z_next": torch.from_numpy(self.Z[i+1]).float(),
                "cti": torch.tensor(float(self.cti[i])),
                "ghi": torch.tensor(float(self.ghi[i])),
                "cov": torch.from_numpy(self.cov[i]).float() if self.cov.shape[1] > 0 else torch.zeros(C_DIM)}

tr_ds = LatentSeqDataset(data["train"])
va_ds = LatentSeqDataset(data["val"])

def train_sde_seed(seed, epochs=60):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM, drift_h=256, diff_h=64).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    dl = DataLoader(tr_ds, batch_size=128, shuffle=True, drop_last=True)
    best = float("inf")
    for ep in range(epochs):
        model.train()
        for b in dl:
            z = b["z_t"].to(DEVICE); zn = b["z_next"].to(DEVICE)
            cti = b["cti"].to(DEVICE).unsqueeze(-1); c = b["cov"].to(DEVICE)
            t = torch.zeros(z.shape[0], 1, device=DEVICE)
            l = model.sde_matching_loss(z, zn, t, c, cti)["loss"]
            opt.zero_grad(); l.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    torch.save(model.state_dict(), CHECKPOINT_DIR / f"sde_seed{seed}.pt")
    return model

def train_score_seed(seed, epochs=30):
    torch.manual_seed(seed); np.random.seed(seed)
    model = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, h=256, blocks=2, steps=100).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-4)
    dl = DataLoader(tr_ds, batch_size=256, shuffle=True, drop_last=True)
    for ep in range(epochs):
        model.train()
        for b in dl:
            z = b["z_t"].to(DEVICE); cti = b["cti"].to(DEVICE).unsqueeze(-1)
            c = b["cov"].to(DEVICE); g = b["ghi"].to(DEVICE).unsqueeze(-1)
            l = model.training_loss(g, z, cti, c)["loss"]
            opt.zero_grad(); l.backward(); opt.step()
    torch.save(model.state_dict(), CHECKPOINT_DIR / f"score_seed{seed}.pt")
    return model

def eval_pair(sde_m, score_m):
    sde_m.eval(); score_m.eval()
    te = data["test"]; res = {}
    for h in HORIZONS:
        yt, ys, rm, cti_eval = [], [], [], []
        for i in range(0, N_EVAL, 32):
            idx = list(range(i, min(i + 32, N_EVAL)))
            z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
            c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
            cti = torch.from_numpy(te["cti"][idx]).float().unsqueeze(-1).to(DEVICE)
            with torch.no_grad():
                endp = solve_sde_horizons(sde_m, z0, [h], c, cti, N=N_SAMPLES)[h]
                B, N, d = endp.shape
                g = score_m.sample(endp.view(B*N, d),
                                  cti.unsqueeze(1).expand(B, N, -1).reshape(B*N, -1),
                                  c.unsqueeze(1).expand(B, N, -1).reshape(B*N, -1), n=1).squeeze(-1).view(B, N).cpu().numpy()
            for k, ii in enumerate(idx):
                j = ii + h
                if j < len(te["ghi"]):
                    yt.append(te["ghi"][j]); ys.append(g[k]); rm.append(te["ramp"][j]); cti_eval.append(te["cti"][ii])
        m = all_metrics(np.array(yt), np.array(ys), is_ramp=np.array(rm))
        m["horizon_min"] = HORIZON_MIN[h]
        res[h] = m
        res[h]["_y_true"] = np.array(yt); res[h]["_y_samples"] = np.array(ys)
        res[h]["_cti"] = np.array(cti_eval)
    return res


## 1. Multi-seed training & evaluation (seeds 123, 456)

In [ ]:
all_results = {}
# Seed 42 was used in Notebook 2 — reload that result
s42 = pd.read_csv(RESULTS_DIR / "solar_sde_main_results.csv")
all_results[42] = s42.set_index("horizon_min").to_dict(orient="index")

for seed in [123, 456]:
    print(f"\n=== Training seed {seed} ===")
    t0 = time.time()
    sde_s = train_sde_seed(seed, epochs=60)
    score_s = train_score_seed(seed, epochs=30)
    print(f"  Training done in {(time.time()-t0)/60:.1f} min")
    res_s = eval_pair(sde_s, score_s)
    all_results[seed] = {HORIZON_MIN[h]: {k: v for k, v in r.items() if not k.startswith("_")}
                        for h, r in res_s.items()}
    # Also save raw samples for analysis (only seed 42 for space — use loaded result)
    if seed == 42: pass

# Build 3-seed aggregate table
seeds = [42, 123, 456]
metrics_keys = ["crps", "rmse", "mae", "picp", "pinaw", "ramp_crps"]
agg_rows = []
for hmin in HORIZON_MIN.values():
    for mk in metrics_keys:
        vals = [all_results[s][hmin].get(mk, np.nan) for s in seeds]
        agg_rows.append({"horizon_min": hmin, "metric": mk,
                         "mean": float(np.nanmean(vals)), "std": float(np.nanstd(vals))})
agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(RESULTS_DIR / "multi_seed_aggregated.csv", index=False)

print("=" * 70)
print("3-SEED AGGREGATED RESULTS (SolarSDE)")
print("=" * 70)
pivot = agg_df.pivot(index="horizon_min", columns="metric", values="mean").round(3)
pivot_std = agg_df.pivot(index="horizon_min", columns="metric", values="std").round(3)
print("Means:"); print(pivot.to_string())
print("\nStds:"); print(pivot_std.to_string())


## 2. Re-generate per-point results for CTI analysis (seed 42)

In [ ]:
# Load seed 42 models (from Notebook 2 checkpoints)
sde = LatentNeuralSDE(z_dim=Z_DIM, c_dim=C_DIM, drift_h=256, diff_h=64).to(DEVICE)
sde.load_state_dict(torch.load(CHECKPOINT_DIR / "sde_best.pt", map_location=DEVICE))
score = CondScoreDecoder(z_dim=Z_DIM, c_dim=C_DIM, h=256, blocks=2, steps=100).to(DEVICE)
score.load_state_dict(torch.load(CHECKPOINT_DIR / "score_best.pt", map_location=DEVICE))

print("Generating per-point predictions (seed 42) for 10-min horizon ...")
H_ANALYSIS = 60  # 10 min
te = data["test"]
yt, ys, cti_eval, ramp_eval = [], [], [], []
for i in range(0, N_EVAL, 32):
    idx = list(range(i, min(i + 32, N_EVAL)))
    z0 = torch.from_numpy(te["Z"][idx]).float().to(DEVICE)
    c = torch.from_numpy(te["cov"][idx]).float().to(DEVICE)
    cti = torch.from_numpy(te["cti"][idx]).float().unsqueeze(-1).to(DEVICE)
    with torch.no_grad():
        endp = solve_sde_horizons(sde, z0, [H_ANALYSIS], c, cti, N=N_SAMPLES)[H_ANALYSIS]
        B, N, d = endp.shape
        g = score.sample(endp.view(B*N, d),
                        cti.unsqueeze(1).expand(B, N, -1).reshape(B*N, -1),
                        c.unsqueeze(1).expand(B, N, -1).reshape(B*N, -1), n=1).squeeze(-1).view(B, N).cpu().numpy()
    for k, ii in enumerate(idx):
        j = ii + H_ANALYSIS
        if j < len(te["ghi"]):
            yt.append(te["ghi"][j]); ys.append(g[k]); cti_eval.append(te["cti"][ii]); ramp_eval.append(te["ramp"][j])

yt = np.array(yt); ys = np.array(ys); cti_eval = np.array(cti_eval); ramp_eval = np.array(ramp_eval)
crps_vals = crps_empirical(yt, ys)
print(f"Evaluation points: {len(yt)}, mean CRPS: {crps_vals.mean():.3f}")


## 3. CTI analysis

In [ ]:
# 3a: Spearman correlation CTI vs GHI variability
print("3a: CTI vs GHI variability")
window = 6
ghi_test = data["test"]["ghi"]
ghi_std = np.zeros_like(ghi_test)
for t in range(window, len(ghi_test)):
    ghi_std[t] = np.std(ghi_test[t - window:t])
cti_test = data["test"]["cti"]
mask = (cti_test > 0) & (ghi_std > 0)
rho, pv = stats.spearmanr(cti_test[mask], ghi_std[mask])
print(f"  Spearman ρ = {rho:.3f}, p-value = {pv:.2e} (N={mask.sum()})")

# 3b: CRPS by CTI quartile
print("\n3b: CRPS by CTI quartile")
qs = np.quantile(cti_eval[cti_eval > 0], np.linspace(0, 1, 5))
cti_quartile_stats = []
for i in range(4):
    m = (cti_eval >= qs[i]) & (cti_eval < qs[i+1] if i < 3 else cti_eval <= qs[i+1])
    if m.sum() > 0:
        cti_mean = float(cti_eval[m].mean())
        crps_mean = float(crps_vals[m].mean())
        cti_quartile_stats.append({"quartile": i+1, "cti_mean": cti_mean, "crps_mean": crps_mean, "n": int(m.sum())})
        print(f"  Q{i+1}: CTI={cti_mean:.4f}, CRPS={crps_mean:.3f}, N={m.sum()}")

# 3c: K-means regime clustering
print("\n3c: CTI regime clustering")
valid_cti = cti_test[cti_test > 0].reshape(-1, 1)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_valid = km.fit_predict(valid_cti)
centers = sorted(km.cluster_centers_.flatten())
regime_names = ["Clear", "Thin Cloud", "Broken Cloud", "Overcast"]
regime_stats = []
for i, name in enumerate(regime_names):
    mask_c = labels_valid == np.argsort(km.cluster_centers_.flatten())[i]
    ghi_ss = data["test"]["ghi"][cti_test > 0][mask_c]
    regime_stats.append({
        "regime": name, "cti_center": float(centers[i]), "n": int(mask_c.sum()),
        "ghi_mean": float(ghi_ss.mean()), "ghi_std": float(ghi_ss.std()),
    })
    print(f"  {name}: CTI={centers[i]:.4f}, GHI mean={ghi_ss.mean():.1f}±{ghi_ss.std():.1f}, N={mask_c.sum()}")

cti_analysis = {
    "spearman_rho": float(rho),
    "spearman_pvalue": float(pv),
    "quartile_stats": cti_quartile_stats,
    "regime_stats": regime_stats,
}
(RESULTS_DIR / "cti_analysis.json").write_text(json.dumps(cti_analysis, indent=2))


## 4. Economic value simulation

In [ ]:
# Simulate grid operator making reserve commitments at 95% quantile of forecast distribution
reserve_quantile = 0.95
reserve_cost = 50.0    # $/MWh reserve
penalty = 1000.0       # $/MWh under-reserve shortfall
decision_min = 5
plant_mw = 1000.0
dt_sec = 10

steps_per_decision = (decision_min * 60) // dt_sec

def simulate_cost(y_true, y_samples):
    reserve = np.quantile(y_samples, reserve_quantile, axis=1)
    idx = np.arange(0, len(y_true), steps_per_decision)
    rc = pc = 0.0
    for i in idx:
        res_mw = (reserve[i] / 1000.0) * plant_mw
        act_mw = (y_true[i] / 1000.0) * plant_mw
        hours = decision_min / 60
        rc += res_mw * reserve_cost * hours
        if act_mw > res_mw: pc += (act_mw - res_mw) * penalty * hours
    total = rc + pc
    # Extrapolate to annual
    test_hours = len(y_true) * dt_sec / 3600
    annual_scale = 365.25 * 12 / test_hours
    return {"reserve_cost": float(rc), "penalty_cost": float(pc), "total_cost": float(total),
            "annual_total": float(total * annual_scale), "annual_per_gw": float(total * annual_scale / (plant_mw / 1000))}

# Simulate for SolarSDE
cost_solar = simulate_cost(yt, ys)
print("SolarSDE costs:"); [print(f"  {k}: ${v:,.0f}") for k, v in cost_solar.items()]

# Simulate for persistence baseline (load per-point from baselines notebook output not saved)
# As proxy: simulate persistence on yt with calibrated Gaussian noise
tr_ghi = data["train"]["ghi"]
r_h = tr_ghi[H_ANALYSIS:] - tr_ghi[:-H_ANALYSIS]
pers_std = float(np.std(r_h))
rng = np.random.default_rng(42)
ys_pers = np.zeros_like(ys)
for i in range(len(yt)):
    pt = data["test"]["ghi"][i] if i < len(data["test"]["ghi"]) else yt[i]
    ys_pers[i] = np.clip(pt + rng.normal(0, pers_std, size=N_SAMPLES), 0, None)
cost_pers = simulate_cost(yt, ys_pers)
print("\nPersistence costs:"); [print(f"  {k}: ${v:,.0f}") for k, v in cost_pers.items()]

savings = {
    "annual_savings_per_gw": cost_pers["annual_per_gw"] - cost_solar["annual_per_gw"],
    "relative_savings_pct": (cost_pers["total_cost"] - cost_solar["total_cost"]) / cost_pers["total_cost"] * 100,
}
print(f"\nAnnual savings: ${savings['annual_savings_per_gw']/1e6:.2f}M/GW/yr")
print(f"Relative cost reduction: {savings['relative_savings_pct']:.1f}%")

econ_summary = {"solar_sde": cost_solar, "persistence": cost_pers, "savings": savings}
(RESULTS_DIR / "economic_value.json").write_text(json.dumps(econ_summary, indent=2))


## 5. Reliability diagram + PIT histogram

In [ ]:
# PIT values
pit = np.mean(ys <= yt[:, None], axis=1)

# Reliability: observed coverage vs nominal
levels = np.arange(0.1, 1.0, 0.1)
observed = []
for L in levels:
    lo = np.quantile(ys, (1 - L) / 2, axis=1)
    hi = np.quantile(ys, 1 - (1 - L) / 2, axis=1)
    observed.append(float(((yt >= lo) & (yt <= hi)).mean()))
reliability = {"nominal": levels.tolist(), "observed": observed}
(RESULTS_DIR / "reliability_data.json").write_text(json.dumps(reliability, indent=2))
print("Reliability:");
for n, o in zip(levels, observed): print(f"  nominal={n:.1f} -> observed={o:.3f}")


## 6. Generate all figures

In [ ]:
# Figure 2: CRPS vs horizon
fig, ax = plt.subplots(figsize=(8, 5))
hs = sorted(HORIZON_MIN.values())
ax.plot(hs, [all_results[42][h]["crps"] for h in hs], "o-", color="#e74c3c", label="SolarSDE (seed 42)", linewidth=2.5)
try:
    # Overlay baselines if available
    comb = pd.read_csv(RESULTS_DIR / "main_results_combined.csv")
    for model, color in [("persistence", "#95a5a6"), ("smart_persistence", "#7f8c8d"),
                         ("lstm", "#3498db"), ("mc_dropout", "#2980b9"), ("csdi", "#9b59b6")]:
        sub = comb[comb["model"] == model].sort_values("horizon_min")
        if not sub.empty:
            ax.plot(sub["horizon_min"], sub["crps"], "o-", color=color, label=model, linewidth=1.2)
except Exception as e:
    print(f"Could not overlay baselines: {e}")
ax.set_xlabel("Forecast Horizon (min)"); ax.set_ylabel("CRPS (W/m²)")
ax.set_title("Probabilistic Forecast Performance"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig2_crps_vs_horizon.pdf", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved fig2_crps_vs_horizon.pdf")

# Figure 3a: CTI vs GHI std scatter
mask = (cti_test > 0) & (ghi_std > 0)
fig, ax = plt.subplots(figsize=(6, 5))
idx_plot = np.random.choice(np.where(mask)[0], min(3000, mask.sum()), replace=False)
ax.scatter(cti_test[idx_plot], ghi_std[idx_plot], alpha=0.3, s=5, c="#3498db")
ax.set_xlabel("Cloud Turbulence Index (CTI)")
ax.set_ylabel("GHI Variability (1-min rolling std, W/m²)")
ax.set_title(f"CTI vs Irradiance Variability\n(Spearman ρ = {rho:.3f})")
ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig3a_cti_scatter.pdf", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved fig3a_cti_scatter.pdf")

# Figure 3b: CRPS by CTI quartile
fig, ax = plt.subplots(figsize=(6, 4))
labels = [f"Q{s['quartile']}\n(CTI={s['cti_mean']:.3f})" for s in cti_quartile_stats]
vals = [s["crps_mean"] for s in cti_quartile_stats]
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.85, len(labels)))
ax.bar(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xlabel("CTI Quartile"); ax.set_ylabel("Mean CRPS (W/m²)")
ax.set_title("Forecast Error by Cloud Turbulence")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig3b_crps_by_cti.pdf", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved fig3b_crps_by_cti.pdf")

# Figure 5: Reliability diagram
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration", alpha=0.5)
ax.plot(levels, observed, "o-", color="#e74c3c", linewidth=2.5, markersize=6, label="SolarSDE")
ax.set_xlabel("Nominal Coverage"); ax.set_ylabel("Observed Coverage")
ax.set_title("Calibration Reliability Diagram"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_aspect("equal"); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig5_reliability.pdf", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved fig5_reliability.pdf")

# PIT histogram
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(pit, bins=10, range=(0, 1), density=True, color="#3498db", edgecolor="white", alpha=0.8)
ax.axhline(y=1.0, color="red", linestyle="--", label="Uniform reference")
ax.set_xlabel("PIT Value"); ax.set_ylabel("Density"); ax.set_title("PIT Histogram (SolarSDE)")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_pit_histogram.pdf", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved fig_pit_histogram.pdf")

# Figure 6: Economic value
fig, ax = plt.subplots(figsize=(7, 4))
m_names = ["Persistence", "SolarSDE"]
costs = [cost_pers["annual_per_gw"] / 1e6, cost_solar["annual_per_gw"] / 1e6]
colors = ["#95a5a6", "#e74c3c"]
bars = ax.bar(m_names, costs, color=colors, edgecolor="white", linewidth=0.5)
ax.set_ylabel("Annual Reserve Cost ($M / GW)")
ax.set_title(f"Economic Value (Savings: ${savings['annual_savings_per_gw']/1e6:.2f}M/GW/yr)")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig6_economic_value.pdf", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved fig6_economic_value.pdf")

print(f"\nAll figures saved to: {FIGURES_DIR}")
for f in sorted(FIGURES_DIR.glob("*.pdf")):
    print(f"  {f.name}: {f.stat().st_size/1024:.1f} KB")


## 7. Final paper tables

In [ ]:
# Table 1: Main results at 10-min horizon
try:
    comb = pd.read_csv(RESULTS_DIR / "main_results_combined.csv")
    t1 = comb[comb["horizon_min"] == 10].copy()
    t1 = t1[["model", "crps", "rmse", "mae", "picp", "pinaw", "ramp_crps", "skill_vs_persistence"]]
    t1.to_csv(RESULTS_DIR / "paper_table1_main.csv", index=False)
    print("Paper Table 1 (main results @ 10-min):")
    print(t1.to_string(index=False))
except Exception as e:
    print(f"Skipping table 1: {e}")

# Table 2: Ablation at 10-min horizon
try:
    abl = pd.read_csv(RESULTS_DIR / "ablation_results.csv")
    t2 = abl[abl["horizon_min"] == 10][["variant", "crps", "rmse", "picp", "pinaw"]].copy()
    t2.to_csv(RESULTS_DIR / "paper_table2_ablation.csv", index=False)
    print("\nPaper Table 2 (ablation @ 10-min):")
    print(t2.to_string(index=False))
except Exception as e:
    print(f"Skipping table 2: {e}")


In [ ]:
# ==== Zip outputs and prepare download ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} -> {zip_path}")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive size: {size_mb:.1f} MB")

if IN_COLAB:
    from google.colab import files
    try:
        files.download(str(zip_path))
        print("Download triggered (check browser).")
    except Exception as e:
        print(f"Auto-download unavailable: {e}. Download manually from {zip_path}")
else:
    print(f"Archive at: {zip_path}  — download via Kaggle output tab or file browser.")


In [ ]:
print("=" * 70); print("NOTEBOOK 5 COMPLETE — FULL EXPERIMENT PIPELINE DONE"); print("=" * 70)
print("All results in:", PERSIST_DIR)
for sub in ["splits", "checkpoints", "results", "latents", "figures"]:
    p = PERSIST_DIR / sub
    if p.exists():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        print(f"  {sub}/: {n} files, {total/1e6:.1f} MB")
